# 01 Platform runtime smoke

Exercise callback retry, heartbeat, local JSONL replay and a bound terminal receipt.

This private `orchestrator_protected` notebook is generated from a reviewed Python template. It receives exact input versions and secrets through Kaggle runtime inputs; no credential is embedded in this notebook or written to its output.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import subprocess
import sys
from pathlib import Path

EXPECTED_SOURCE_SHA256 = '8fbc5444d058be867b762d2d8640780d61c188ea32bb37a2ae1034263ba5523b'
RUNTIME_CONTRACT = 'my-data-hub-platform-runtime-smoke.v1'
PIN_CONTRACT = {'schema': 'my-data-hub-notebook-execution-pins/v1', 'notebook': '01-platform-runtime-smoke', 'supported_python_series': '3.12', 'kaggle_runtime_image_identity': 'required-immutable-sha256-at-launch', 'input_dataset_versions': 'required-exact-numeric-private-refs-at-launch', 'immutable_assets': ['my_data_hub_wheel_sha256', 'primary_source_sha256'], 'output_contract': 'my-data-hub-platform-runtime-smoke.v1', 'model': None, 'privacy': 'private', 'resource_class': 'orchestrator_protected', 'cleanup_retention_policy': {'cleanup_receipt_required': True, 'notebook_resource': 'orchestrator_protected_until_owner_supersedes', 'run_outputs': 'retain_until_terminal_receipt_then_control_policy', 'task_owned_inputs': 'claim_bound_delete_after_terminal_or_expiry'}}
pin_path = Path(os.environ.get('MY_DATA_HUB_EXECUTION_PINS_PATH', ''))
expected_pin_sha = os.environ.get('MY_DATA_HUB_EXECUTION_PINS_SHA256', '')
if not pin_path.is_file() or not re.fullmatch(r'[a-f0-9]{64}', expected_pin_sha):
    raise RuntimeError('hashed execution pins manifest is required')
pin_bytes = pin_path.read_bytes()
if hashlib.sha256(pin_bytes).hexdigest() != expected_pin_sha:
    raise RuntimeError('execution pins manifest hash mismatch')
pins = json.loads(pin_bytes)
required_pin_keys = {
    'schema', 'notebook', 'python_version', 'kaggle_runtime_image_identity',
    'input_dataset_versions', 'immutable_asset_sha256s', 'output_contract',
    'model', 'privacy', 'resource_class', 'cleanup_retention_policy',
}
if not isinstance(pins, dict) or set(pins) != required_pin_keys:
    raise RuntimeError('execution pins manifest keys differ from the exact contract')
if pins['schema'] != PIN_CONTRACT['schema'] or pins['notebook'] != PIN_CONTRACT['notebook']:
    raise RuntimeError('execution pins manifest targets a different notebook contract')
python_version = platform.python_version()
if (pins['python_version'] != python_version or
        not python_version.startswith(PIN_CONTRACT['supported_python_series'] + '.')):
    raise RuntimeError('CPython patch version differs from execution pins')
image_identity = os.environ.get('MY_DATA_HUB_KAGGLE_RUNTIME_IMAGE_IDENTITY', '')
if (pins['kaggle_runtime_image_identity'] != image_identity or
        not re.fullmatch(r'[^@\s]+@sha256:[a-f0-9]{64}', image_identity)):
    raise RuntimeError('immutable Kaggle runtime image identity is required')
dataset_versions = pins['input_dataset_versions']
if (not isinstance(dataset_versions, list) or not dataset_versions or
        any(not isinstance(ref, str) or not re.fullmatch(
            r'[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+/[1-9][0-9]*', ref
        ) for ref in dataset_versions) or
        len(dataset_versions) != len(set(dataset_versions))):
    raise RuntimeError('exact numeric input Dataset versions are required')
try:
    observed_dataset_versions = json.loads(
        os.environ.get('MY_DATA_HUB_INPUT_DATASET_VERSIONS_JSON', '')
    )
except json.JSONDecodeError as exc:
    raise RuntimeError('observed input Dataset versions are required') from exc
if observed_dataset_versions != dataset_versions:
    raise RuntimeError('attached input Dataset versions differ from execution pins')
if os.environ.get('MY_DATA_HUB_NOTEBOOK_IS_PRIVATE') != 'true':
    raise RuntimeError('operational notebook must be provider-confirmed private')
for key in ('output_contract', 'model', 'privacy', 'resource_class', 'cleanup_retention_policy'):
    if pins[key] != PIN_CONTRACT[key]:
        raise RuntimeError(f'execution pins {key} differs from the generated contract')
wheel = Path(os.environ.get('MY_DATA_HUB_WHEEL_PATH', ''))
if not wheel.is_file() or wheel.suffix != '.whl':
    raise RuntimeError('exact private my-data-hub wheel input is required')
expected_wheel_sha = os.environ.get('MY_DATA_HUB_WHEEL_SHA256', '')
if (len(expected_wheel_sha) != 64 or 
        hashlib.sha256(wheel.read_bytes()).hexdigest() != expected_wheel_sha):
    raise RuntimeError('my-data-hub wheel hash mismatch')
expected_assets = {
    'my_data_hub_wheel_sha256': expected_wheel_sha,
    'primary_source_sha256': EXPECTED_SOURCE_SHA256,
}
if pins['immutable_asset_sha256s'] != expected_assets:
    raise RuntimeError('immutable dependency/source asset hashes differ from execution pins')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--disable-pip-version-check', str(wheel)],
    check=True,
)

In [ ]:
PRIMARY_SOURCE = '"""Primary source for the private Kaggle platform/runtime smoke notebook."""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nfrom my_data_hub.hashing import canonical_json_bytes, sha256_value\nfrom my_data_hub.runtime_sdk import RuntimeClient, RuntimeEventType\n\n\ndef _required(name: str) -> str:\n    value = os.environ.get(name, "")\n    if not value:\n        raise RuntimeError(f"required runtime value is absent: {name}")\n    return value\n\n\ndef main() -> int:\n    output = Path("/kaggle/working")\n    client = RuntimeClient(\n        callback_url=_required("MY_DATA_HUB_CALLBACK_URL"),\n        run_secret=_required("MY_DATA_HUB_RUN_SECRET"),\n        run_id=_required("MY_DATA_HUB_RUN_ID"),\n        attempt_id=_required("MY_DATA_HUB_ATTEMPT_ID"),\n        service_instance_id=_required("MY_DATA_HUB_SERVICE_INSTANCE_ID"),\n        source_identity=_required("MY_DATA_HUB_SOURCE_IDENTITY"),\n        source_version=_required("MY_DATA_HUB_SOURCE_VERSION"),\n        epoch=int(_required("MY_DATA_HUB_EPOCH")),\n        spool_path=output / "runtime-events.jsonl",\n        heartbeat_interval_seconds=5.0,\n    )\n    client.replay_pending()\n    client.emit(RuntimeEventType.RUNTIME_STARTED, phase="smoke", status="running")\n    client.emit(RuntimeEventType.RUNTIME_HEARTBEAT, phase="smoke", status="healthy", data={"step": 1})\n    receipt = {\n        "schema_version": "my-data-hub-run-receipt.v1",\n        "task_run_id": client.run_id,\n        "provider_ref": _required("MY_DATA_HUB_SOURCE_IDENTITY"),\n        "source_version": _required("MY_DATA_HUB_SOURCE_VERSION"),\n        "source_sha256": _required("MY_DATA_HUB_SOURCE_SHA256"),\n        "terminal_state": "complete",\n        "output_sha256": sha256_value({"runtime_contract": "platform-runtime-smoke.v1"}),\n    }\n    (output / "my-data-hub-run-receipt.json").write_bytes(canonical_json_bytes(receipt))\n    client.emit(RuntimeEventType.RUNTIME_TERMINAL, phase="smoke", status="complete", data={"ok": True})\n    return 0\n'
if hashlib.sha256(PRIMARY_SOURCE.encode()).hexdigest() != EXPECTED_SOURCE_SHA256:
    raise RuntimeError('embedded primary source hash mismatch')
exec(compile(PRIMARY_SOURCE, '<my-data-hub-primary-source>', 'exec'), globals())

In [ ]:
raise SystemExit(globals()['main']())